Tic Tac Toe
---
Two players against each other

<img style="float:left" src="board.png" alt="drawing" width="200"/>

In [7]:
import numpy as np
import pickle

BOARD_ROWS = 3
BOARD_COLS = 3

### Board State
---
Reflect & Judge the state

2 players p1 and p2; p1 uses symbol1 = 1 and p2 uses symbol2 = -1, vacancy as 0

In [ ]:
class State:
    def __init__(self, p1, p2): # p1 = minmax or alphabeta player, p2 = human player
        self.board = np.zeros((BOARD_ROWS, BOARD_COLS), dtype=int)
        # numpy array to represent the board, 0 = empty, 1 = player1, -1 = player2
        self.p1 = p1 # first player - starts each time
        self.p2 = p2 # second player
        self.isEnd = False
        # init p1 plays first
        self.playerSymbol = 1
    
    # returns -1 is p2 wins, 1 if p1 wins and 0 for tie and sets isEnd to True
    def winner(self):
        # check rows
        for i in range(BOARD_ROWS):
            if sum(self.board[i, :]) == 3:
                self.isEnd = True
                return 1
            if sum(self.board[i, :]) == -3:
                self.isEnd = True
                return -1
        # check columns
        for i in range(BOARD_COLS):
            if sum(self.board[:, i]) == 3:
                self.isEnd = True
                return 1
            if sum(self.board[:, i]) == -3:
                self.isEnd = True
                return -1
        # check diagonals
        diag_sum1 = sum([self.board[i, i] for i in range(BOARD_COLS)])
        diag_sum2 = sum([self.board[i, BOARD_COLS-i-1] for i in range(BOARD_COLS)])
        if (diag_sum1 == 3 or diag_sum2 == 3):
            self.isEnd  = True
            return 1
        elif (diag_sum1 == -3 or diag_sum2 == -3):
            self.isEnd  = True
            return -1
        
        # tie if no available positions
        if len(self.availablePositions()) == 0:
            self.isEnd = True
            return 0
        # not end
        self.isEnd = False
        return None # game not over yet
    
    # returns a list of available positions on the board
    def availablePositions(self):
        positions = []
        for i in range(BOARD_ROWS):
            for j in range(BOARD_COLS):
                if self.board[i, j] == 0:
                    positions.append((i, j))  # need to be tuple
        return positions
    
    # updates the board state and switches to the next player
    def updateState(self, position):
        self.board[position] = self.playerSymbol
        # switch to another player
        self.playerSymbol = -1 if self.playerSymbol == 1 else 1
    
    # board reset
    def reset(self):
        self.board = np.zeros((BOARD_ROWS, BOARD_COLS))
        self.isEnd = False
        self.playerSymbol = 1 # first player starts first with 1 
    

    # minmax play player1 = minmax/alphabeta, player = human
    def play(self):

        while not self.isEnd:
            # Player 1 - computer 
            p1_action = self.p1.chooseAction(self.board, self.playerSymbol)
            # take action and upate board state
            self.updateState(p1_action) # switches the playerSymbol to -1 for player2
            # check board status if it is end
            self.showBoard()

            win = self.winner()
            if win is not None: # if the game has ended, print the result and return  
                if win == 1:
                    print(self.p1.name, "wins")
                elif win == -1:
                    print(self.p2.name, "wins")
                else:
                    print("Tie")
                return
                #self.reset() # starts a new game
                
            # player2 = human
            p2_action = self.p2.chooseAction(self.board, self.playerSymbol)
            self.updateState(p2_action) # switches the playerSymbol to 1 for player1
            self.showBoard()

            win = self.winner()
            if win is not None:   # if the game has ended, print the result and return     
                if win == 1:
                    print(self.p1.name, "wins")
                elif win == -1:
                    print(self.p2.name, "wins")
                else:
                    print("Tie")
                return
                #self.reset()
                
    # displays the board in a human-readable format
    def showBoard(self):
        # p1: x  p2: o
        for i in range(0, BOARD_ROWS):
            print('-------------')
            out = '| '
            for j in range(0, BOARD_COLS):
                if self.board[i, j] == 1:
                    token = 'x'
                if self.board[i, j] == -1:
                    token = 'o'
                if self.board[i, j] == 0:
                    token = ' '
                out += token + ' | '
            print(out)
        print('-------------')    

In [12]:
class HumanPlayer:
    def __init__(self, name):
        self.name = name 
    
    def chooseAction(self, current_board, playerSymbol): 
        # need playerSymbol to determine which player is playing, 
        # but not used in this function, used in the class State 
        # to update the board state and switch to the next player
        positions = self.available_positions(current_board)
        while True:
            row = int(input("Input your action row:"))
            col = int(input("Input your action col:"))
            action = (row, col)
            if action in positions:
                return action
    
    # returns a list of available positions on the board as tuples (all positions == 0)
    def available_positions(self, board): 
        positions = []
        for i in range(BOARD_ROWS):
            for j in range(BOARD_COLS):
                if board[i, j] == 0:
                    positions.append((i, j))  # need to be tuple
        return positions


In [10]:
# get a unique hash of current board state string representation of the board
# board is a numpy array of shape (3, 3) representing the tic-tac-toe board
def getHash(board): 
    boardHash = str(board.reshape(BOARD_COLS*BOARD_ROWS)) 
    # self.board.reshape(BOARD_COLS*BOARD_ROWS) flattens the 2D board into a 1D array, then converts to string
    # example: [[ 1  0 -1] [ 0  1  0] [-1  0  1]] -> '100010-101' 
    # how does this work? 1st row: 1 0 -1, 2nd row: 0 1 0, 3rd row: -1 0 1, 
    # then concatenates all elements into a single string = '[ 1  0 -1  0  1  0 -1  0  1]'
    # this becomes the unique hash for this board state.
    return boardHash

In [11]:
import numpy as np
sample_board = np.array([[ 1,  0, -1], [ 0,  1,  0], [-1,  0,  1]])
s_board = getHash(sample_board)
print(s_board)

[ 1  0 -1  0  1  0 -1  0  1]


In [ ]:
from copy import deepcopy # makes a deep copy of the board so that we can simulate moves without affecting the original board

class MinMaxPlayer:
    def __init__(self, name):
        self.name = name
        self.states_values = {} # store the value of each state in a dictionary, 
        # where the key is the hash of the state and the value is the value of that state

    # use this to store the value of a state in the states_values dictionary
    def store_value(self, state, value):
        hash_state = getHash(state)
        self.states_values[hash_state] = value

    def chooseAction(self, board, playerSymbol):
        current_board = deepcopy(board) # make a copy of the board so that we can simulate moves without affecting the original board
        
        value,chosen_position = self.minmax(board, playerSymbol) # chosen_position can be None, or a tuple
        
        current_board[chosen_position[0]][chosen_position[1]] = playerSymbol # update the board with the chosen position
        self.store_value(current_board, value) # current_board is the board state after the move, value is the value of that state
        
        return chosen_position
        
    def minmax(self, board, playerSymbol):
        value, chosen_position = self.max_value(board, playerSymbol)
        return value, chosen_position
    
    def max_value(self, board, playerSymbol):
        val_current = self.winner(board, playerSymbol)
        if val_current is not None: # val_current is a terminal state
            return val_current, None
        
        # current_board is not a terminal state
        valid_positions = self.available_positions(board)
        value = -100
        chosen_position = None

        new_board = deepcopy(board) # make a copy of the board so that we can simulate 
                                    # moves without affecting the original board
        for pos in valid_positions:
            new_board[pos[0]][pos[1]] = playerSymbol # MAKE MOVE ON NEW BOARD
            
            # get new valid_positions
            min_val, _ = self.min_value(new_board,(-1)playerSymbol)
            if min_val > value:
                value = min_val
                chosen_position = pos
            
            new_board[pos[0]][pos[1]] = 0 # UNDO THE MOVE
        
        return value, chosen_position
    
    # complete min_value method to return the minimum value and the chosen 
    # position for the min player
    def min_value(self, current_board, playerSymbol):
        # add code
        pass
        # return value, chosen_position
    
    # complete winner method to return the winner of the game,
    # 1 if player 1 wins, -1 if player 2 wins,
    # 0 if tie, None if the game is not over yet
    def winner(self, board, playerSymbol): # Modified winner method
        # rows
        for i in range(BOARD_ROWS):
            if sum(board[i, :]) == 3: #* playerSymbol:
                return 1
            if sum(board[i, :]) == -3: #* playerSymbol:
                return -1
        # cols
        for i in range(BOARD_COLS):
            if sum(board[:, i]) == 3: # * playerSymbol:
                return 1
            if sum(board[:, i]) == -3: # * playerSymbol:
                return -1
        # diagonals
        diag_sum1 = sum([board[i, i] for i in range(BOARD_COLS)])
        diag_sum2 = sum([board[i, BOARD_COLS-i-1] for i in range(BOARD_COLS)])
        if diag_sum1 == 3 or diag_sum2 == 3:
            return 1
        elif diag_sum1 == -3 or diag_sum2 == -3:
            return -1  
        # tie
        if len(self.available_positions(board)) == 0:
            return 0
        
        # not end
        return None
    
    # returns a list of available positions on the board as tuples (all positions == 0)
    def available_positions(self, board): 
        positions = []
        for i in range(BOARD_ROWS):
            for j in range(BOARD_COLS):
                if board[i, j] == 0:
                    positions.append((i, j))  # need to be tuple
        return positions
    
    def reset(self):
        self.states = []



### Human vs Computer

In [ ]:
p1 = MinMaxPlayer("computer")
#p1.loadPolicy("policy_p1")

ph = HumanPlayer("human")

st = State(p1,ph)
st.play_against_human()

-------------
| x |   |   | 
-------------
|   |   |   | 
-------------
|   |   |   | 
-------------
